# 专家级复现标准：同一数据跑出同一图

> 本 Notebook 对应文章：`010_S0_专家级复现标准：同一数据跑出同一图[通用][进阶].md`
>
> 目标：逐步执行文章中的所有代码，验证可行性

## 环境准备

首次运行时，请先安装依赖包

In [ ]:
# 安装依赖（首次运行时取消注释）
# !pip install scanpy squidpy matplotlib

In [ ]:
import scanpy as sc
import squidpy as sq
import numpy as np
import pandas as pd

# 打印版本信息
print(f"Scanpy: {sc.__version__}")
print(f"Squidpy: {sq.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

# 保存到日志
with open('session_info.txt', 'w') as f:
    sc.logging.print_versions(file=f)

**输出示例**：

**验收标准**：
- ✅ 环境配置文件已提交到代码仓库
- ✅ 每次分析都记录版本信息
- ✅ 团队成员能用配置文件重建环境

## 2. 随机种子：控制随机过程

### 哪些步骤涉及随机性

空间转录组分析中，很多步骤依赖随机算法：

- **PCA**：初始化随机向量
- **UMAP**：随机初始化嵌入坐标
- **Leiden/Louvain 聚类**：随机选择起始节点
- **空间邻域构建**：某些算法使用随机采样

不设置随机种子 (Random Seed)，每次运行结果都会略有不同。

### 全局设置随机种子

In [ ]:
import numpy as np
import random
import scanpy as sc

# 设置全局随机种子
SEED = 42

np.random.seed(SEED)
random.seed(SEED)
sc.settings.seed = SEED

print(f"Random seed set to {SEED}")

### 在关键步骤显式设置

即使设置了全局种子，仍建议在关键步骤显式指定：

In [ ]:
# PCA
sc.tl.pca(adata, random_state=SEED)

# UMAP
sc.tl.umap(adata, random_state=SEED)

# Leiden 聚类
sc.tl.leiden(adata, random_state=SEED)

# 邻域图构建
sc.pp.neighbors(adata, random_state=SEED)

### 诊断：验证随机种子有效性

In [ ]:
import scanpy as sc
import numpy as np

# 加载数据
adata = sc.datasets.visium_sge(sample_id="V1_Mouse_Brain_Sagittal_Posterior")

# 第一次运行
np.random.seed(42)
sc.pp.neighbors(adata, random_state=42)
sc.tl.leiden(adata, random_state=42)
clusters_1 = adata.obs['leiden'].copy()

# 第二次运行（重新加载数据）
adata = sc.datasets.visium_sge(sample_id="V1_Mouse_Brain_Sagittal_Posterior")
np.random.seed(42)
sc.pp.neighbors(adata, random_state=42)
sc.tl.leiden(adata, random_state=42)
clusters_2 = adata.obs['leiden'].copy()

# 检查一致性
assert (clusters_1 == clusters_2).all(), "Clustering results differ!"
print("✓ Clustering is reproducible")

### 负对照：不设置随机种子的后果

为了验证随机种子的必要性，我们设置一个负对照 (Negative Control)：不设置随机种子，观察结果的变化。

In [ ]:
import scanpy as sc
import numpy as np

# 加载数据
adata = sc.datasets.visium_sge(sample_id="V1_Mouse_Brain_Sagittal_Posterior")

# 第一次运行（不设置种子）
sc.pp.neighbors(adata)
sc.tl.leiden(adata)
clusters_1 = adata.obs['leiden'].copy()

# 第二次运行（重新加载数据，不设置种子）
adata = sc.datasets.visium_sge(sample_id="V1_Mouse_Brain_Sagittal_Posterior")
sc.pp.neighbors(adata)
sc.tl.leiden(adata)
clusters_2 = adata.obs['leiden'].copy()

# 比较结果
match_rate = (clusters_1 == clusters_2).sum() / len(clusters_1)
print(f"Cluster assignment match rate: {match_rate:.2%}")
# 输出示例: Cluster assignment match rate: 87.3%
# 说明：即使大部分一致，仍有 12.7% 的 spot 聚类结果不同

**关键发现**：不设置随机种子时，即使是同一数据，聚类结果也会有 10-15% 的差异。这种不确定性会导致下游分析（如差异基因检测、空间域识别）完全不同。

**验收标准**：
- ✅ 代码开头设置全局随机种子
- ✅ 关键函数显式传入 `random_state`
- ✅ 多次运行得到完全相同的结果

## 3. 参数记录：完整记录分析配置

### 为什么需要记录参数

三个月后，你可能忘记：

- 质控时用的线粒体基因阈值是 10% 还是 20%？
- 归一化 (Normalization) 用的是 `total` 还是 `median`？
- 空间邻域半径是 100 µm 还是 150 µm？

**解决方案**：用配置文件管理所有参数。

### 方案 A：YAML 配置文件

**在代码中使用**：

In [ ]:
import yaml

# 加载配置
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# 使用配置
sc.pp.filter_cells(adata, min_genes=config['quality_control']['min_genes'])
sc.pp.filter_cells(adata, min_counts=config['quality_control']['min_counts'])

# 归一化
sc.pp.normalize_total(
    adata, 
    target_sum=config['normalization']['target_sum']
)
if config['normalization']['log_transform']:
    sc.pp.log1p(adata)

# 聚类
sc.pp.neighbors(
    adata, 
    n_neighbors=config['clustering']['n_neighbors'],
    n_pcs=config['clustering']['n_pcs']
)
sc.tl.leiden(adata, resolution=config['clustering']['resolution'])

### 方案 B：在代码中定义配置字典

In [ ]:
# 配置参数
CONFIG = {
    'qc': {
        'min_genes': 200,
        'min_counts': 500,
        'max_mito_pct': 20.0
    },
    'normalization': {
        'target_sum': 10000
    },
    'clustering': {
        'n_neighbors': 15,
        'n_pcs': 50,
        'resolution': 0.8
    }
}

# 保存配置到文件
import json
with open('analysis_config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)

### 诊断：参数追踪

In [ ]:
# 在 AnnData 对象中记录参数
adata.uns['analysis_params'] = {
    'qc_min_genes': 200,
    'qc_max_mito': 20.0,
    'norm_target_sum': 10000,
    'clustering_resolution': 0.8,
    'random_seed': 42
}

# 保存到文件
adata.write('results/adata_processed.h5ad')

# 读取时可以查看参数
adata_loaded = sc.read_h5ad('results/adata_processed.h5ad')
print(adata_loaded.uns['analysis_params'])

**验收标准**：
- ✅ 所有关键参数记录在配置文件中
- ✅ 配置文件与代码一起提交到版本控制
- ✅ 参数变化时更新配置文件并记录原因

## 4. 版本管理：追踪代码变化

### Git 基础工作流

### 关键文件清单

**必须提交**：
- ✅ 分析脚本（`.py`, `.ipynb`）
- ✅ 配置文件（`config.yaml`, `environment.yml`）
- ✅ README 文件（README 是项目说明文档，说明如何复现）

**不要提交**：
- ❌ 原始数据文件（`.h5ad`, `.csv`）- 太大，用数据仓库
- ❌ 中间结果（`results/`）- 可以重新生成
- ❌ 图片（`figures/`）- 可以重新生成

### 完整的 README 示例

# Create conda environment
conda env create -f environment.yml
conda activate st-analysis

python 01_quality_control.py
python 02_normalization.py
python 03_clustering.py
python 04_spatial_analysis.py

**验收标准**：
- ✅ 代码托管在 Git 仓库
- ✅ README 包含完整的复现步骤
- ✅ 提交信息清晰描述每次变更

## 完整示例：可复现的空间分析流程

下面是一个完整的可复现分析示例：

In [ ]:
"""
Reproducible Spatial Transcriptomics Analysis
Author: Your Name
Date: 2026-04-08
Environment: See environment.yml
"""

import scanpy as sc
import squidpy as sq
import numpy as np
import pandas as pd
import yaml

# ============================================================
# 1. Setup: Random Seed & Configuration
# ============================================================

SEED = 42
np.random.seed(SEED)
sc.settings.seed = SEED

# Load configuration
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Print session info
print("=" * 60)
print("Session Information")
print("=" * 60)
print(f"Scanpy: {sc.__version__}")
print(f"Squidpy: {sq.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Random Seed: {SEED}")
print("=" * 60)

# ============================================================
# 2. Load Data
# ============================================================

adata = sc.datasets.visium_sge(sample_id="V1_Mouse_Brain_Sagittal_Posterior")
print(f"Loaded data: {adata.shape[0]} spots × {adata.shape[1]} genes")

# ============================================================
# 3. Quality Control
# ============================================================

# Calculate QC metrics
sc.pp.calculate_qc_metrics(adata, inplace=True)

# Filter based on config
sc.pp.filter_cells(
    adata, 
    min_genes=config['quality_control']['min_genes']
)
sc.pp.filter_cells(
    adata, 
    min_counts=config['quality_control']['min_counts']
)

print(f"After QC: {adata.shape[0]} spots × {adata.shape[1]} genes")

# ============================================================
# 4. Normalization
# ============================================================

sc.pp.normalize_total(
    adata, 
    target_sum=config['normalization']['target_sum']
)
sc.pp.log1p(adata)

# ============================================================
# 5. Feature Selection & Dimensionality Reduction
# ============================================================

sc.pp.highly_variable_genes(adata, n_top_genes=2000)
sc.pp.pca(adata, random_state=SEED)
sc.pp.neighbors(
    adata, 
    n_neighbors=config['clustering']['n_neighbors'],
    n_pcs=config['clustering']['n_pcs'],
    random_state=SEED
)
sc.tl.umap(adata, random_state=SEED)

# ============================================================
# 6. Clustering
# ============================================================

sc.tl.leiden(
    adata, 
    resolution=config['clustering']['resolution'],
    random_state=SEED
)

print(f"Identified {adata.obs['leiden'].nunique()} clusters")

# ============================================================
# 7. Spatial Analysis
# ============================================================

sq.gr.spatial_neighbors(
    adata, 
    coord_type=config['spatial']['coord_type'],
    radius=config['spatial']['neighborhood_radius']
)

# ============================================================
# 8. Save Results & Parameters
# ============================================================

# Record parameters in AnnData
adata.uns['analysis_params'] = {
    'random_seed': SEED,
    'scanpy_version': sc.__version__,
    'squidpy_version': sq.__version__,
    'config': config,
    'analysis_date': '2026-04-08'
}

# Save processed data
adata.write('results/adata_processed.h5ad')

print("Analysis complete. Results saved to results/adata_processed.h5ad")

### 配置文件（config.yaml）

### 环境文件（environment.yml）

## 验证可复现性：自检清单

运行以下测试，确保你的分析是可复现的：

### 测试 1：环境可复现

**预期结果**：版本号与 environment.yml 一致。

### 测试 2：结果可复现

In [ ]:
# 运行两次分析，比较结果
import scanpy as sc
import numpy as np

# 第一次
np.random.seed(42)
adata1 = sc.datasets.visium_sge(sample_id="V1_Mouse_Brain_Sagittal_Posterior")
sc.pp.normalize_total(adata1)
sc.pp.log1p(adata1)
sc.pp.pca(adata1, random_state=42)
sc.pp.neighbors(adata1, random_state=42)
sc.tl.leiden(adata1, random_state=42)

# 第二次
np.random.seed(42)
adata2 = sc.datasets.visium_sge(sample_id="V1_Mouse_Brain_Sagittal_Posterior")
sc.pp.normalize_total(adata2)
sc.pp.log1p(adata2)
sc.pp.pca(adata2, random_state=42)
sc.pp.neighbors(adata2, random_state=42)
sc.tl.leiden(adata2, random_state=42)

# 比较聚类结果
assert (adata1.obs['leiden'] == adata2.obs['leiden']).all()
print("✓ Results are reproducible")

**预期结果**：断言通过，无报错。

### 测试 3：跨机器可复现

**预期结果**：不同机器上生成的文件哈希值相同。

## 常见问题与解决方案

### 问题 1：conda 环境太大，安装慢

**症状**：`environment.yml` 包含几百个包，安装需要很长时间。

**解决方案**：只锁定核心包，让 conda 自动解析依赖。

### 问题 2：随机种子设置了，结果还是不一致

**可能原因**：
- 某些函数没有 `random_state` 参数
- 多线程/多进程导致的不确定性
- 硬件差异（GPU 计算）

**解决方案**：

In [ ]:
# 禁用多线程
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

# 设置所有随机源
import numpy as np
import random
import scanpy as sc

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
sc.settings.seed = SEED

### 问题 3：数据文件太大，无法提交到 Git

**解决方案**：使用数据仓库（Zenodo, Figshare）或 Git Large File Storage (LFS，Git 大文件存储扩展)。

### 问题 4：Jupyter Notebook 难以版本控制

**解决方案**：使用 `nbconvert` 转换为 Python 脚本。

或使用 `jupytext` 同步 notebook 和脚本：

## 边界声明

本文介绍的方法能够确保：

✅ **能保证的**：
- 相同环境、相同代码、相同数据 → 相同结果
- 团队成员能复现你的分析
- 审稿人能验证你的结果

❌ **不能保证的**：
- 跨平台完全一致（Windows vs Linux 可能有细微差异）
- 跨硬件完全一致（GPU 计算可能有浮点误差）
- 跨版本完全一致（Scanpy 2.0 可能打破兼容性）

**建议**：
- 在论文中明确说明软件版本和运行环境
- 提供 Docker 镜像以实现更强的可复现性
- 定期测试代码在新版本软件上的兼容性

## 验收清单

完成本文学习后，你应该能够：

- [ ] 创建并导出 conda 环境配置文件
- [ ] 在代码中正确设置随机种子
- [ ] 使用配置文件管理分析参数
- [ ] 用 Git 管理代码版本
- [ ] 编写完整的 README 说明复现步骤
- [ ] 验证分析结果的可复现性
- [ ] 识别并解决常见的可复现性问题

## 下一篇预告

下一篇文章将介绍**质量控制的完整流程**，包括：

- 如何识别低质量 spot
- 线粒体基因比例的阈值选择
- 组织内外 spot 的区分
- 批次效应 (Batch Effect) 的诊断

可复现性是基础，质量控制是第一道关卡。我们下期见。

---

**关键要点回顾**：

1. **环境锁定**：用 `environment.yml` 固定软件版本
2. **随机种子**：在全局和关键步骤设置 `random_state`
3. **参数记录**：用 YAML 配置文件管理所有参数
4. **版本管理**：用 Git 追踪代码变化，README 说明复现步骤

可复现性不是额外负担，而是节省时间的投资。一次设置，终身受益。